In [7]:
# ============================================================
# 01_data_preprocessing.ipynb
# KNHANES 2020–2024 Data Preprocessing Pipeline
#
# Paper: Adverse Selection Risk and Counterfactual Explanation
#        Guardrails in Healthcare AI Underwriting:
#        A Quantitative Governance Evaluation under
#        Korea's AI Basic Act
#
# Data Source: Korea National Health and Nutrition Examination
#              Survey (KNHANES), Korea Disease Control and
#              Prevention Agency (KDCA), 2020–2024
# Note: Raw data files are not included in this repository.
#       Access: https://knhanes.kdca.go.kr
# ============================================================


# ─────────────────────────────────────────────
# Cell 1 | Library Imports
# ─────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import pyreadstat

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
pd.set_option('display.max_colwidth', None)

print("Libraries loaded successfully.")


# ─────────────────────────────────────────────
# Cell 2 | Path Configuration
# ─────────────────────────────────────────────
BASE_DATA = os.path.join('..', 'data')
BASE_OUT  = os.path.join('..', 'outputs')
os.makedirs(BASE_OUT, exist_ok=True)

SAS_FILES = {
    2020: os.path.join(BASE_DATA, 'hn20_all.sas7bdat'),
    2021: os.path.join(BASE_DATA, 'hn21_all.sas7bdat'),
    2022: os.path.join(BASE_DATA, 'hn22_all.sas7bdat'),
    2023: os.path.join(BASE_DATA, 'hn23_all.sas7bdat'),
    2024: os.path.join(BASE_DATA, 'hn24_all.sas7bdat'),
}

print("Path configuration:")
for year, path in SAS_FILES.items():
    status = '✅' if os.path.exists(path) else '❌ NOT FOUND'
    print(f"  {year}: {path}  {status}")


# ─────────────────────────────────────────────
# Cell 3 | Variable Definition
# ─────────────────────────────────────────────
# Design principle: only ACTIONABLE variables are included
# as model features. Non-actionable variables are retained
# solely for RAG stratification (age, sex) or dropped entirely.
#
# CFE constraint taxonomy:
#   phi_imm  — immutable (cannot be changed by the applicant)
#   phi_caus — causally inconsistent (derived / redundant)
#   phi_cost — actionable within a realistic time horizon
#
# EXCLUDED (non-actionable):
#   age, sex          — phi_imm: demographic, immutable
#   edu               — phi_imm: fixed in adulthood
#   incm, ho_incm     — phi_imm: income not actionable short-term
#   BE3_71, BE3_81    — phi_imm: occupation-dependent PA
#   BP1, mh_stress    — phi_imm: psychological state, not directly actionable
#   HE_obe            — phi_caus: derived from BMI / weight (redundant)
#   BO1_1, BO1_2,
#   BO1_3             — phi_imm: past weight-change records, retroactive
#
# NOTE: age & sex are retained in the saved CSV files
#       for downstream RAG (Recourse Accessibility Gap) analysis.

# ── Identifiers & stratification variables
key = ['ID', 'year', 'sex', 'age']

# ── Shared actionable categorical features (phi_cost)
cat_shared = [
    'BD1_11',     # Alcohol drinking frequency (12+)
    'BD2_1',      # Alcohol amount per occasion (12+)
    'BS3_1',      # Current cigarette smoking status (adult)
    'BE3_75',     # High-intensity PA: leisure
    'BE3_91',     # PA: transportation
    'pa_aerobic', # Aerobic PA practice rate
    'L_BR_FQ',    # Breakfast frequency per week (past year)
    'BH1',        # Health checkup status (adult)
]

# ── Shared actionable continuous features (phi_cost)
num_shared = [
    'HE_BMI',  # Body Mass Index (kg/m²)
    'HE_wc',   # Waist circumference (cm)
    'HE_wt',   # Body weight (kg)
]

# ── Disease-specific actionable features
# Hypertension: dietary sodium & potassium (phi_cost)
# Leak-excluded: HE_sbp, HE_dbp, HE_HPdr, DI1_2
htn_specific_num = ['N_NA', 'N_K']
htn_specific_cat = []

# Diabetes: macronutrient intake (phi_cost)
# Leak-excluded: HE_glu, HE_HbA1c, HE_DMdr
# Removed (phi_imm): BO1_1, BO1_2, BO1_3 — past weight change
dm_specific_num  = ['N_SUGAR', 'N_CHO', 'N_EN']
dm_specific_cat  = []

# Dyslipidemia: fat & dietary cholesterol intake (phi_cost)
# Leak-excluded: HE_chol, HE_TG, HE_HDL_st2, HE_LDL_drct, DI2_2
dys_specific_num = ['N_FAT', 'N_CHOL']
dys_specific_cat = []

# ── Target variables
# Encoding (consistent across all three diseases):
#   Borderline / pre-disease stage → excluded (NaN)
#   HE_HP       : 1=Normal→0 | 2=Pre-HTN→NaN | 3=HTN→1
#   HE_DM_HbA1c : 1=Normal→0 | 2=Pre-DM→NaN  | 3=DM→1
#   HE_HCHOL    : 0=Normal→0 | 1=Dyslipidemia→1 (already binary)
TARGET_VARS = ['HE_HP', 'HE_DM_HbA1c', 'HE_HCHOL']

# ── Full variable pool per disease
ALL_HTN = (key + cat_shared + htn_specific_cat +
           num_shared + htn_specific_num + ['HE_HP'])
ALL_DM  = (key + cat_shared + dm_specific_cat +
           num_shared + dm_specific_num  + ['HE_DM_HbA1c'])
ALL_DYS = (key + cat_shared + dys_specific_cat +
           num_shared + dys_specific_num + ['HE_HCHOL'])

ALL_VARS_UNION = sorted(set(ALL_HTN + ALL_DM + ALL_DYS))

print("Variable summary (actionable features only):")
print(f"  Shared categorical  : {len(cat_shared)}")
print(f"  Shared continuous   : {len(num_shared)}")
print(f"  HTN-specific        : {len(htn_specific_num + htn_specific_cat)}")
print(f"  DM-specific         : {len(dm_specific_num  + dm_specific_cat)}")
print(f"  DYS-specific        : {len(dys_specific_num + dys_specific_cat)}")
print(f"  Targets             : {len(TARGET_VARS)}")
print(f"  Union pool (load)   : {len(ALL_VARS_UNION)}")
print(f"\nNon-actionable variables excluded from features:")
print(f"  phi_imm  : age, sex, edu, incm, ho_incm")
print(f"  phi_imm  : BE3_71, BE3_81 (occupation-dependent PA)")
print(f"  phi_imm  : BP1, mh_stress (psychological state)")
print(f"  phi_caus : HE_obe (derived from BMI/weight)")
print(f"  phi_imm  : BO1_1, BO1_2, BO1_3 (past weight change)")
print(f"  Note     : age & sex retained in CSV for RAG stratification")


# ─────────────────────────────────────────────
# Cell 4 | Load Raw SAS Files (2020–2024)
# ─────────────────────────────────────────────
print("Loading KNHANES raw data files...")

frames = []
for year, path in SAS_FILES.items():
    df_year, _ = pyreadstat.read_sas7bdat(path)
    df_year['year'] = float(year)

    available = [v for v in ALL_VARS_UNION if v in df_year.columns]
    missing   = [v for v in ALL_VARS_UNION if v not in df_year.columns]

    df_year = df_year[available].copy()
    for v in missing:
        df_year[v] = np.nan

    frames.append(df_year)
    print(f"  {year}: {df_year.shape[0]:,} rows | "
          f"missing vars: {missing if missing else 'none'}")

df_raw = pd.concat(frames, axis=0).reset_index(drop=True)
print(f"\nMerged shape  : {df_raw.shape}")
print(f"Survey years  : "
      f"{sorted(df_raw['year'].dropna().unique().astype(int))}")


# ─────────────────────────────────────────────
# Cell 5 | Target Variable Encoding
# ─────────────────────────────────────────────
# Pre-disease / borderline stages → NaN (excluded).
# This approach is consistent with the prior diabetes paper
# and ensures clean binary classification across all diseases.

df_raw['HE_HP']        = df_raw['HE_HP'].map({1.0: 0, 3.0: 1})
df_raw['HE_DM_HbA1c']  = df_raw['HE_DM_HbA1c'].map({1.0: 0, 3.0: 1})
# HE_HCHOL is already binary (0/1); no remapping needed

print("Target variable distributions after encoding:")
for tgt in TARGET_VARS:
    n0  = (df_raw[tgt] == 0).sum()
    n1  = (df_raw[tgt] == 1).sum()
    nan = df_raw[tgt].isna().sum()
    pos_rate = f"{n1/(n0+n1)*100:.1f}%" if (n0+n1) > 0 else "N/A"
    print(f"\n  {tgt}:")
    print(f"    0 (negative) : {n0:,}")
    print(f"    1 (positive) : {n1:,}  ({pos_rate})")
    print(f"    NaN (excl.)  : {nan:,}")


# ─────────────────────────────────────────────
# Cell 6 | Build Per-Disease DataFrames
# ─────────────────────────────────────────────
def build_disease_df(df, var_list, target_col):
    """
    Select variables, keep valid binary target rows,
    and restrict to adults aged 19 and older.
    """
    df_sub = df[var_list].copy()
    df_sub = df_sub[df_sub[target_col].isin([0, 1])]
    df_sub = df_sub[df_sub['age'] >= 19]
    return df_sub.reset_index(drop=True)

df_htn = build_disease_df(df_raw, ALL_HTN, 'HE_HP')
df_dm  = build_disease_df(df_raw, ALL_DM,  'HE_DM_HbA1c')
df_dys = build_disease_df(df_raw, ALL_DYS, 'HE_HCHOL')

print("After valid-target + adult (19+) filter:")
print(f"  Hypertension  : {len(df_htn):,} rows")
print(f"  Diabetes      : {len(df_dm):,} rows")
print(f"  Dyslipidemia  : {len(df_dys):,} rows")


# ─────────────────────────────────────────────
# Cell 7 | Missing Value Report & Complete-Case
# ─────────────────────────────────────────────
def missing_report(df, label):
    print(f"\n{'='*55}")
    print(f"  Missing Value Report — {label}")
    print(f"{'='*55}")
    miss = df.isnull().sum()
    pct  = (miss / len(df) * 100).round(2)
    rpt  = (pd.DataFrame({'Missing N': miss, 'Missing %': pct})
              .query('`Missing N` > 0')
              .sort_values('Missing %', ascending=False))
    print(rpt.to_string())

missing_report(df_htn, 'Hypertension')
missing_report(df_dm,  'Diabetes')
missing_report(df_dys, 'Dyslipidemia')

df_htn = df_htn.dropna().reset_index(drop=True)
df_dm  = df_dm.dropna().reset_index(drop=True)
df_dys = df_dys.dropna().reset_index(drop=True)

print("\nAfter complete-case analysis:")
print(f"  Hypertension : {len(df_htn):,} rows")
print(f"  Diabetes     : {len(df_dm):,} rows")
print(f"  Dyslipidemia : {len(df_dys):,} rows")


# ─────────────────────────────────────────────
# Cell 8 | Data Type Casting
# ─────────────────────────────────────────────
def cast_types(df, cat_cols, num_cols, target_col):
    df[target_col] = df[target_col].astype(int)
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype(int)
    for c in num_cols:
        if c in df.columns:
            df[c] = df[c].astype(float)
    return df

htn_cat = cat_shared + htn_specific_cat
htn_num = num_shared + htn_specific_num
dm_cat  = cat_shared + dm_specific_cat
dm_num  = num_shared + dm_specific_num
dys_cat = cat_shared + dys_specific_cat
dys_num = num_shared + dys_specific_num

df_htn = cast_types(df_htn, htn_cat, htn_num, 'HE_HP')
df_dm  = cast_types(df_dm,  dm_cat,  dm_num,  'HE_DM_HbA1c')
df_dys = cast_types(df_dys, dys_cat, dys_num, 'HE_HCHOL')

print("Type casting complete.")


# ─────────────────────────────────────────────
# Cell 9 | Outlier Removal (Continuous Variables)
# ─────────────────────────────────────────────
# Rule: exclude rows where any continuous variable falls
#       outside [max(0, mean − 3SD), mean + 3SD].
# A physical lower bound of 0 is enforced for intake and
# anthropometric variables to prevent nonsensical negatives.

def remove_outliers(df, num_cols, label):
    print(f"\n{'='*55}")
    print(f"  Outlier Removal — {label}")
    print(f"{'='*55}")
    before = len(df)
    for col in num_cols:
        if col not in df.columns:
            continue
        mean  = df[col].mean()
        std   = df[col].std()
        lower = max(0.0, mean - 3 * std)
        upper = mean + 3 * std
        removed = ((df[col] < lower) | (df[col] > upper)).sum()
        df = df[(df[col] >= lower) & (df[col] <= upper)]
        print(f"  {col:12s} | [{lower:9.2f}, {upper:9.2f}] "
              f"| removed {removed:4d} rows")
    df = df.reset_index(drop=True)
    print(f"\n  Rows: {before:,} → {len(df):,}  "
          f"(removed {before - len(df):,})")
    return df

df_htn = remove_outliers(df_htn, htn_num, 'Hypertension')
df_dm  = remove_outliers(df_dm,  dm_num,  'Diabetes')
df_dys = remove_outliers(df_dys, dys_num, 'Dyslipidemia')


# ─────────────────────────────────────────────
# Cell 10 | Age Group Derivation (RAG Analysis)
# ─────────────────────────────────────────────
# Three strata aligned with insurance underwriting lifecycle:
#   young   : 19–44  (low awareness, early-stage risk)
#   middle  : 45–64  (core underwriting segment)
#   elderly : 65+    (high prevalence, equity concern)

AGE_BINS   = [18, 44, 64, 120]
AGE_LABELS = ['young', 'middle', 'elderly']

for df in [df_htn, df_dm, df_dys]:
    df['age_group'] = pd.cut(
        df['age'],
        bins=AGE_BINS,
        labels=AGE_LABELS,
        right=True
    )

print("Age group distribution:")
for df, label in [(df_htn, 'Hypertension'),
                  (df_dm,  'Diabetes'),
                  (df_dys, 'Dyslipidemia')]:
    print(f"\n  {label}:")
    print(df['age_group'].value_counts()
            .sort_index().to_string())


# ─────────────────────────────────────────────
# Cell 11 | Class Imbalance Report
# ─────────────────────────────────────────────
print("Class imbalance summary:")
print(f"{'Disease':<16} {'N(0)':>8} {'N(1)':>8} "
      f"{'Pos%':>7} {'scale_pos_weight':>18}")
print("-" * 62)

for df, tgt, label in [
    (df_htn, 'HE_HP',       'Hypertension'),
    (df_dm,  'HE_DM_HbA1c', 'Diabetes'),
    (df_dys, 'HE_HCHOL',    'Dyslipidemia'),
]:
    n0  = (df[tgt] == 0).sum()
    n1  = (df[tgt] == 1).sum()
    spw = round(n0 / n1, 4)
    print(f"  {label:<14} {n0:>8,} {n1:>8,} "
          f"{n1/(n0+n1)*100:>6.1f}% {spw:>18.4f}")


# ─────────────────────────────────────────────
# Cell 12 | Sex & Age-Group Split
# ─────────────────────────────────────────────
def split_and_report(df, tgt, label):
    splits = {}
    for sex_val, sex_lbl in [(1, 'male'), (2, 'female')]:
        df_s = df[df['sex'] == sex_val].reset_index(drop=True)
        splits[sex_lbl] = df_s
        pos = (df_s[tgt] == 1).sum()
        print(f"    {sex_lbl:7s}: {len(df_s):,}  "
              f"(positive: {pos:,}, "
              f"{pos/len(df_s)*100:.1f}%)")
    print(f"    Age-group breakdown:")
    ag = df.groupby('age_group', observed=True)[tgt].agg(
        Total='count', Positive='sum'
    )
    ag['Pos%'] = (ag['Positive'] / ag['Total'] * 100).round(1)
    print(ag.to_string())
    return splits

print("=== Final Dataset Summary ===")
splits_htn, splits_dm, splits_dys = {}, {}, {}

for df, tgt, label, store in [
    (df_htn, 'HE_HP',       'Hypertension', splits_htn),
    (df_dm,  'HE_DM_HbA1c', 'Diabetes',     splits_dm),
    (df_dys, 'HE_HCHOL',    'Dyslipidemia', splits_dys),
]:
    print(f"\n  [{label}]  total: {len(df):,}")
    store.update(split_and_report(df, tgt, label))


# ─────────────────────────────────────────────
# Cell 13 | Save Preprocessed Data
# ─────────────────────────────────────────────
save_map = {
    'htn_total.csv' : df_htn,
    'htn_male.csv'  : splits_htn['male'],
    'htn_female.csv': splits_htn['female'],
    'dm_total.csv'  : df_dm,
    'dm_male.csv'   : splits_dm['male'],
    'dm_female.csv' : splits_dm['female'],
    'dys_total.csv' : df_dys,
    'dys_male.csv'  : splits_dys['male'],
    'dys_female.csv': splits_dys['female'],
}

for fname, df in save_map.items():
    path = os.path.join(BASE_OUT, fname)
    df.to_csv(path, index=False)
    print(f"  Saved: {fname}  ({len(df):,} rows)")

print("\nAll preprocessed files saved to outputs/")

Libraries loaded successfully.
Path configuration:
  2020: ..\data\hn20_all.sas7bdat  ✅
  2021: ..\data\hn21_all.sas7bdat  ✅
  2022: ..\data\hn22_all.sas7bdat  ✅
  2023: ..\data\hn23_all.sas7bdat  ✅
  2024: ..\data\hn24_all.sas7bdat  ✅
Variable summary (actionable features only):
  Shared categorical  : 8
  Shared continuous   : 3
  HTN-specific        : 2
  DM-specific         : 3
  DYS-specific        : 2
  Targets             : 3
  Union pool (load)   : 25

Non-actionable variables excluded from features:
  phi_imm  : age, sex, edu, incm, ho_incm
  phi_imm  : BE3_71, BE3_81 (occupation-dependent PA)
  phi_imm  : BP1, mh_stress (psychological state)
  phi_caus : HE_obe (derived from BMI/weight)
  phi_imm  : BO1_1, BO1_2, BO1_3 (past weight change)
  Note     : age & sex retained in CSV for RAG stratification
Loading KNHANES raw data files...
  2020: 7,359 rows | missing vars: none
  2021: 7,090 rows | missing vars: none
  2022: 6,265 rows | missing vars: none
  2023: 6,929 rows | mis